In [2]:
from sqlalchemy import create_engine
import pandas as pd

pd.options.display.max_columns = None

db_config = {
 'user': 'practicum_student', # username
 'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7', # password
 'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
 'port': 5432, 
 'db': 'data-analyst-final-project-db'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [4]:
# Vista rápida de cada tabla clave (ajusta LIMIT si lo deseas)
for tbl in ["books", "authors", "publishers", "ratings", "reviews"]:
    try:
        print(f"\n===== {tbl} =====")
        df = pd.io.sql.read_sql(f"SELECT * FROM {tbl} LIMIT 5;", con=engine)
        display(df)
    except Exception as e:
        print(f"⚠️ No se pudo leer {tbl}: {e}")


===== books =====


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268



===== authors =====


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd



===== publishers =====


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company



===== ratings =====


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2



===== reviews =====


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


In [6]:
# Seleccionar columnas específicas
q = """
    SELECT * 
    FROM books
    LIMIT 5
"""
res = pd.io.sql.read_sql(q, con=engine)
display(res)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


In [25]:
# Seleccionar columnas específicas
q = """
    SELECT 
        title,
        publication_date,
        num_pages
    FROM books
    WHERE
        publication_date >= '2000-01-01' 
        AND num_pages < 1000
    ORDER BY
        publication_date ASC
    LIMIT 5
"""
res = pd.io.sql.read_sql(q, con=engine)
display(res)

,title,publication_date,num_pages
0,A Room of One's Own,2000-01-01,112
1,Shopgirl,2000-01-01,130
2,Angels Flight (Harry Bosch #6; Harry Bosch Un...,2000-01-05,454
3,The Light Fantastic (Discworld #2; Rincewind #2),2000-02-02,277
4,The Gods Themselves,2000-02-10,288


In [30]:
# Seleccionar columnas específicas
q = """
    SELECT  COUNT(*)
    FROM books
"""
res = pd.io.sql.read_sql(q, con=engine)
display(res)

,count
0,1000


In [35]:
# Seleccionar columnas específicas
q = """
    SELECT 
        b.title,
        b.publication_date,
        b.num_pages,
        a.author,
        AVG(r.rating) 
    FROM books b
    INNER JOIN authors a 
        ON b.author_id = a.author_id
    INNER JOIN  ratings r
        ON b.book_id = r.book_id
    WHERE
        publication_date >= '2000-01-01' 
        AND num_pages < 1000
    GROUP BY
        b.title,
        b.publication_date,
        b.num_pages,
        a.author
    HAVING
        AVG(r.rating) >= 4.0
    ORDER BY
        publication_date ASC
    LIMIT 10
"""
res = pd.io.sql.read_sql(q, con=engine)
display(res)

,title,publication_date,num_pages,author,avg
0,Shopgirl,2000-01-01,130,Steve Martin,4.000000
1,A Room of One's Own,2000-01-01,112,Virginia Woolf,4.000000
2,Angels Flight (Harry Bosch #6; Harry Bosch Un...,2000-01-05,454,Michael Connelly,4.500000
3,The Gods Themselves,2000-02-10,288,Isaac Asimov,4.500000
4,The Firm,2000-02-15,76,Robin Waterfield/John Grisham,4.052632
5,Traveling Mercies: Some Thoughts on Faith,2000-02-15,275,Anne Lamott,4.000000
6,My Ántonia (Great Plains Trilogy #3),2000-02-20,232,Willa Cather,4.250000
7,House of Leaves,2000-03-07,705,Mark Z. Danielewski,4.750000
8,Hatchet (Brian's Saga #1),2000-04-01,208,Gary Paulsen,4.200000
9,The Diamond Age: Or A Young Lady's Illustrate...,2000-05-02,499,Neal Stephenson/Pedro Jorge Romero,4.333333
